# 11 Source and data audit
Reads the manifest written by `scripts/prepare_logic_data.py` and shows the checks that gate any training (plan Gate 1).

In [ ]:
import sys, json
from pathlib import Path
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT / "src"))
from moebase import paths

man = json.loads((paths.DATA_DIR / paths.DATA_VERSION / "manifest.json").read_text())
audit = man["audit"]
print("label mismatches:", audit.get("label_mismatches"), "| degenerate:", audit.get("degenerate_clauses"),
      "| file-index mismatches:", audit["file_index_mismatches"])
print("A/B/C train sizes:", audit["systematicity_train_sizes"], "match paper:", audit["systematicity_sizes_match_paper"])
print("upstream:", json.dumps(paths.UPSTREAM, indent=1))

In [ ]:
import pandas as pd
rows = [{"regime": r, "part": p, "n": d["n"], "majority": round(d["majority_label_rate"], 3), "swap_dup": d["swap_dup"],
         "max_tokens": d["max_tokens_pair_plus_sep"]} for r, parts in man["splits"].items() for p, d in parts.items()]
pd.DataFrame(rows)

In [ ]:
import matplotlib.pyplot as plt
fig, ax = plt.subplots(1, 2, figsize=(11, 3.5))
lab = man["splits"]["length"]["train"]["labels"]
ax[0].bar(list(lab), list(lab.values())); ax[0].set_title("train label counts")
for part in ("train", "test_iid", "test_len"):
    lb = man["splits"]["length"][part]["length_bins"]
    ax[1].plot([int(k) for k in lb], list(lb.values()), marker="o", label=part)
ax[1].set_xlabel("operators (max over clauses)"); ax[1].set_yscale("log"); ax[1].legend(); plt.tight_layout()